In [211]:
import gensim.models
from corus import load_lenta
import spacy
import string
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import urllib.request

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Возьмём подготовленные данные из ДЗ №1 с уже проведенной лемматизацией, токенизацией, исключением стоп-слов и знаков препинания

In [3]:
df = pd.read_csv('lenta_ru_news_lemma_csv')


In [4]:
df.head(5)

,title,text,topic,full_text,token_text
0,Столица Таджикистана осталась без света,Столица Таджикистана Душанбе практически полно...,Мир,Столица Таджикистана осталась без света Столиц...,столица таджикистан остаться свет столица тадж...
1,Мотопарад в Москве установил рекорд,Более пяти тысяч человек приняли участие в мот...,Россия,Мотопарад в Москве установил рекорд Более пяти...,мотопарад москва установить рекорд пять тыся...
2,КНДР провела неудачный запуск ракеты в день ро...,Северная Корея попыталась запустить баллистиче...,Силовые структуры,КНДР провела неудачный запуск ракеты в день ро...,кндр провести неудачный запуск ракета день р...
3,Путин поручил рассмотреть вопрос о введении ут...,Президент России Владимир Путин поручил правит...,Бизнес,Путин поручил рассмотреть вопрос о введении ут...,путин поручить рассмотреть вопрос введение у...
4,Поляки отправили в Россию последнюю партию отр...,Последняя партия ядерных отходов из польского ...,Бизнес,Поляки отправили в Россию последнюю партию отр...,поляк отправить россия последний партия отра...


In [5]:
df.shape

(100000, 5)

# 1. Разделим датасет на обучающую, валидационную и тестовую выборки со стратификацией в пропорции 60/20/20. В качестве целевой переменной атрибут topic

In [ ]:
# Разделение на train и остальное
X_train, X_razd, y_train, y_razd = train_test_split(df['token_text'], df['topic'],test_size=0.4,stratify=df['topic'],random_state=42)

# Разделение остального на val и test по 20% от исходного
X_val, X_test, y_val, y_test = train_test_split(X_razd,y_razd,test_size=0.5, stratify=y_razd, random_state=42)

# 2. Обучим word2vec-эмбеддинги с помощью библиотеки gensim - 2 балла


## 2.1 Создадим модель для обучения на данных, объяснить какими значениями инициализированы гиперпараметры модели, и почему

In [7]:
X_train.iloc[0]

'умереть оператор владимир нахабцев воскресение москва 64-ом год жизнь скончаться известный кинооператор владимир нахабцев владимир нахабцев родиться 1938 год окончить вгик окончание институт работать эльдар рязанов сотрудничество владимир нахабцевым снять фильм гусарский баллада беречься автомобиль зигзаг удача ирония судьба лёгкий пар служебный роман гараж лента войти золотой фонд российский кино владимир дмитриевич охотно сотрудничать известный мастер число витаутас жалакявичус сладкий слово свобода сергей соловьёв москва любовь марк захаров мюнхгаузен убить дракон владимир дмитриевич нахабцев народный артист россия лауреат государственный премия доцент вгик член российский академия кинематографический искусство отпевание владимир нахабцева состояться 13 март 11.00 храм воскресение словущего брюсовом переулок похороны пройти 13.00 троекуровский кладбище передавать итар тасс'

In [8]:
train_tokens = [text.split() for text in X_train]

In [9]:
train_tokens[0][:10]

['умереть',
 'оператор',
 'владимир',
 'нахабцев',
 'воскресение',
 'москва',
 '64-ом',
 'год',
 'жизнь',
 'скончаться']

In [10]:
%%time

model = gensim.models.Word2Vec(
    sentences=train_tokens,
    vector_size=300,
    window=5,
    min_count=5,
    sg=1,
    negative=15,
    epochs=30,
    seed=2026
)

CPU times: total: 2h 54min 35s
Wall time: 58min 49s


Взяли следующие гиперпараметры:

1. vector_size=300 - размерность векторов (эмбеддинга) для каждого слова, данное значение является стандартом, увеличивая данный параметр намного больше, это может привести лишь к проклятию размерности, но не улучшить информативное представление вектора

2. window=5 - размер контекстного окна, также является оптимальным решением, учитываем как самые ближайший контекст, так и чуть более широкий


3. min_count=5 - минимальная частота слова для включения в словарь, отсекаем самые редкие слова, опечатки и так далее, которые можно классифицировать как шум. Тем самым мы уменьшим количество эмбеддингов, ускорим обучение, при большом значение параметра >10, > 15 мы потярем много важных слов


4. sg=1 - алгоритм обучения модели Word2Vec, 1 - это skip-gram, то есть предсказываем контекст по слову, sg = 0 - это CBOW, предсказание слово по контексту, skip gram также зачаствую является более предпочтительным выбором для получения эмбеддингов с модели

5. negative=7 - количество негативных примеров для каждого слова, которые показывают для каждого примера, чтобы научить отличать хорошие слова(часто встречаются рядом) от плохих(негативных)
Что это: количество негативных примеров при обучении


6. epochs=30 - количество эпох обучения нейронной сети, с маленьким кол-вом можем получить недообучение, с большим переобучение


7. seed - для воспроизводимости результатов, при каждом обучении будут одинаковые вектора в итоге


## 2.2 Визуально оценим внутреннее (intrinsic) качество получившихся эмбеддингов, используя методы gensim - doesnt_match, most_similar

Посмотрим на слова похожие на слово 'аэропорт'

In [17]:
print(model.wv.most_similar('аэропорт', topn=5))

[('шереметьево', 0.6575104594230652), ('гавань', 0.6407954692840576), ('домодедово', 0.6365079879760742), ('авиагавани', 0.6191274523735046), ('гатвик', 0.6104001998901367)]


Посмотрим на слова похожие на слова похожие одновренно на 'магазин' и 'продукт'

In [ ]:
 
print(model.wv.most_similar(positive=['магазин', 'продукт'], topn=5))

[('товар', 0.6379032135009766), ('супермаркет', 0.5506812930107117), ('мультибрендовых', 0.5323744416236877), ('makro', 0.5243129134178162), ('продуктовый', 0.5221770405769348)]


Посмотрим на слова похожие на слово, которое получается из путин минус германия

In [46]:
 
print(model.wv.most_similar(positive=['путин'], negative = ['германия'], topn=5))

[('владимир', 0.3830786347389221), ('колокольцеву', 0.32864394783973694), ('варлей', 0.3210940361022949), ('медведев', 0.3128989338874817), ('президент', 0.30664992332458496)]


Произведём поиск лишнего слова из предложенных

In [49]:
print(model.wv.doesnt_match(['россия', 'украина', 'беларусь', 'футбол']))

футбол


In [54]:
print(model.wv.doesnt_match(['мфти', 'мгу','итмо','мэи','папа']))

папа


Для такого не слишком огромного корпуса текстов, обученных на модели word2vec, видим, что так или иначе эмбеддинги имеют какую-либо осмысленность и информативность в векторном пространстве

# 3. Загрузим предобученные эмбеддинги из navec и rusvectores 

Загрузим rusvectores обученный НКРЯ и Википедии методом skip-gram

In [187]:
urllib.request.urlretrieve(
    "https://rusvectores.org/static/models/rusvectores4/ruwikiruscorpora/ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz",
    "ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz"
)

('ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz',
 <http.client.HTTPMessage at 0x18753b8d540>)

In [189]:

model_path = 'ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz'
model_rusvec = gensim.models.KeyedVectors.load_word2vec_format(model_path)

Проверим, что всё работает

In [192]:
model_rusvec.most_similar(positive=['москва_NOUN'], topn=10)

[('москва“_NOUN', 0.6156069040298462),
 ('тбилиси_NOUN', 0.6120966076850891),
 ('вологда_NOUN', 0.5830255746841431),
 ('новосибирск_NOUN', 0.5769392251968384),
 ('вечернять_NOUN', 0.5720601677894592),
 ('россия_NOUN', 0.5682430267333984),
 ('урал_NOUN', 0.5679124593734741),
 ('ленинград_NOUN', 0.5559836030006409),
 ('волга_NOUN', 0.547827959060669),
 ('петербург_NOUN', 0.5443581342697144)]

Загрузим navec модель специально предназначенную для новостных текстов

In [129]:

from navec import Navec
path = 'navec_news_v1_1B_250K_300d_100q.tar'
model_navec = Navec.load(path)

Проверим, что модель успешно загружена и выдаёт эмбеддинги

In [130]:
model_navec['марсель'][:10]

array([-0.3116582 , -0.48993498,  0.5017227 ,  0.72642905, -0.15794787,
        0.20088129, -0.07467852,  0.38789082,  0.41766357,  0.20964701],
      dtype=float32)

# 4. Обучим модель sklearn.linear_model.LogisticRegression с тремя вариантами векторизации текстов и сравним их качество между собой на валидационной выборке

Проведём векторизацию текстов по принципу одна новость - одна строка - один объединенный усредненный эмбеддинг токенизированных слов, если слова нет в обученной модели, то оно не учитывается

In [151]:
def avg_embedding_to_train_model(text, model, vector_size=300):
    """
    Преобразование текста в средний эмбеддинг всех слов для обученной модели
    """
    words = text.split()
    vectors = []
    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)



In [152]:

X_train_w2v = np.array([avg_embedding_to_train_model(text, model) for text in X_train])
X_val_w2v = np.array([avg_embedding_to_train_model(text, model) for text in X_val])
X_test_w2v = np.array([avg_embedding_to_train_model(text, model) for text in X_test])

In [160]:
def avg_embedding_to_rusvec(text, model_rusvec, vector_size=300):
    """
    Преобразование текста в средний эмбеддинг всех слов для модели rusvec
    """
    words = text.split()
    vectors = []
    for word in words:
        if word in model_rusvec:
            vectors.append(model_rusvec[word])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)



In [180]:

X_train_rusvec = np.array([avg_embedding_to_rusvec(text, model_rusvec) for text in X_train])
X_val_rusvec = np.array([avg_embedding_to_rusvec(text, model_rusvec) for text in X_val])
X_test_rusvec = np.array([avg_embedding_to_rusvec(text, model_rusvec) for text in X_test])

In [162]:
def avg_embedding_to_navec(text, model_navec, vector_size=300):
    """
    Преобразование текста в средний эмбеддинг всех слов для модели navec
    """
    words = text.split()
    vectors = []
    for word in words:
        if word in model_navec:
            vectors.append(model_navec[word])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)



In [163]:

X_train_navec = np.array([avg_embedding_to_navec(text, model_navec) for text in X_train])
X_val_navec = np.array([avg_embedding_to_navec(text, model_navec) for text in X_val])
X_test_navec = np.array([avg_embedding_to_navec(text, model_navec) for text in X_test])

In [199]:
def train_three_models(X_train, X_val, y_train, y_val, name_model):
    lr = LogisticRegression(max_iter=1200, C=10, random_state=128)
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    print(f"{name_model} - Accuracy: {accuracy:.4f}")
    print(classification_report(y_val, y_pred))
    return accuracy



In [200]:
acc_w2v = train_three_models(X_train_w2v, X_val_w2v, y_train, y_val, "train_Word2Vec")

train_Word2Vec - Accuracy: 0.7999
                   precision    recall  f1-score   support

           Бизнес       0.76      0.74      0.75      1125
      Бывший СССР       0.84      0.86      0.85      1520
              Дом       0.89      0.88      0.88      1520
         Из жизни       0.70      0.71      0.70      1520
   Интернет и СМИ       0.77      0.75      0.76      1520
         Культура       0.86      0.87      0.87      1520
              Мир       0.71      0.75      0.73      1520
  Наука и техника       0.81      0.79      0.80      1520
      Путешествия       0.84      0.84      0.84       974
           Россия       0.66      0.64      0.65      1520
Силовые структуры       0.74      0.75      0.75      1520
            Спорт       0.96      0.96      0.96      1520
         Ценности       0.91      0.90      0.91      1181
        Экономика       0.79      0.77      0.78      1520

         accuracy                           0.80     20000
        macro avg   

In [205]:
acc_rusvec = train_three_models(X_train_rusvec, X_val_rusvec, y_train, y_val, "rusvec")


rusvec - Accuracy: 0.0760
                   precision    recall  f1-score   support

           Бизнес       0.00      0.00      0.00      1125
      Бывший СССР       0.00      0.00      0.00      1520
              Дом       0.00      0.00      0.00      1520
         Из жизни       0.00      0.00      0.00      1520
   Интернет и СМИ       0.00      0.00      0.00      1520
         Культура       0.00      0.00      0.00      1520
              Мир       0.08      1.00      0.14      1520
  Наука и техника       0.00      0.00      0.00      1520
      Путешествия       0.00      0.00      0.00       974
           Россия       0.00      0.00      0.00      1520
Силовые структуры       0.00      0.00      0.00      1520
            Спорт       0.00      0.00      0.00      1520
         Ценности       0.00      0.00      0.00      1181
        Экономика       0.00      0.00      0.00      1520

         accuracy                           0.08     20000
        macro avg       0.01

c:\Users\Marsohodik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Marsohodik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Marsohodik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [207]:
acc_navec = train_three_models(X_train_navec, X_val_navec, y_train, y_val, "navec")

navec - Accuracy: 0.7745
                   precision    recall  f1-score   support

           Бизнес       0.69      0.65      0.67      1125
      Бывший СССР       0.82      0.86      0.84      1520
              Дом       0.87      0.85      0.86      1520
         Из жизни       0.68      0.70      0.69      1520
   Интернет и СМИ       0.75      0.74      0.74      1520
         Культура       0.85      0.87      0.86      1520
              Мир       0.72      0.71      0.72      1520
  Наука и техника       0.79      0.78      0.78      1520
      Путешествия       0.79      0.80      0.80       974
           Россия       0.63      0.60      0.62      1520
Силовые структуры       0.67      0.71      0.69      1520
            Спорт       0.96      0.96      0.96      1520
         Ценности       0.89      0.88      0.88      1181
        Экономика       0.73      0.73      0.73      1520

         accuracy                           0.77     20000
        macro avg       0.77 

In [210]:
print("Финальное сравнение результатов на валидации:")
print(f"Обученный  Word2Vec: {acc_w2v:.4f}")
print(f"Rusvec: {acc_rusvec:.4f}")
print(f"Navec: {acc_navec:.4f}")

Финальное сравнение результатов на валидации:
Обученный  Word2Vec: 0.7999
Rusvec: 0.0760
Navec: 0.7745


##### Таким образом, после получения эмбеддингов для train, test, val выборок с помощью различных моделей и обучения на логистической регрессии, самое лучшее качество было достигнуто на модели Word2Vec обученной на нашем корпусе текстов, думаю это достигнуто из-за того, что данная модель как раз и обучалась на данном корпусе текстов, а потом мы получили усреденные эмбеддинги от этого же корпуса текстов. Самая худшая точность получена с помощью модели RusVec, т.к слова в этой модели размечены частями речи, а наш корпус текстов нет, из-за чего с помощью этой модели мы не можем найти огромное количество наших токенизированных слов в этой модели

# 5. Попробуем улучшить качество модели, взяв для ее обучения лучший набор эмбеддингов и используя его с взвешиванием через tf-idf. То есть, необходимо каждый текст представить в виде взвешенного усреднения эмбеддингов его слов, где весами являются соответствующие коэффициенты tf-idf -

Т.к обученная word2vec модель показала лучший результат, то используем её

Создадим с tf-idf словарь для бучающей выборки

In [212]:
tfidf_vector = TfidfVectorizer()
tfidf_vector.fit(X_train)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


 Каждому слову присвоим индекс

In [ ]:
words = tfidf_vector.get_feature_names_out()
word_to_index = {word: idx for idx, word in enumerate(words)}

Функция для преобразование нашего текста в tf idf взвешенный средний эмбеддинг

In [ ]:
def weighted_embeddings(text, word2vec_model, tfidf_vector, word_to_index, vector_size=300):
    words = text.split() # разделим общую строку наших лематизированных слов на отдельные слова в массив
    vectors = []  # для итоговых эмбеддингов * вес tf idf 
    weights = []  # для весов tf idf 
    tfidf_matrix = tfidf_vector.transform([text])

    for word in words:
        if word in word2vec_model.wv and word in word_to_index:
            idx = word_to_index[word] # берём индекс слова из word_to_index
            weight = tfidf_matrix[0, idx]  # берём вес слова
            vectors.append(word2vec_model.wv[word] * weight) # умножаем эмбеддинг на вес
            weights.append(weight)
    
    if vectors and sum(weights) > 0:  # проверка на то что хотя бы одно слово было найдено и вес есть
        # Вычисляем взвешенное среднее
        weighted_avg = np.sum(vectors, axis=0) / sum(weights)
        return weighted_avg
    else:
        return np.zeros(vector_size)


In [228]:
X_train_w2v_tfidf = np.array([weighted_embeddings(text, model, tfidf_vector, word_to_index) for text in X_train])
X_val_w2v_tfidf = np.array([weighted_embeddings(text, model, tfidf_vector, word_to_index) for text in X_val])

In [244]:
X_test_w2v_tfidf = np.array([weighted_embeddings(text, model, tfidf_vector, word_to_index) for text in X_test])

In [237]:
acc_w2v_and_tfidf = train_three_models(X_train_w2v_tfidf, X_val_w2v_tfidf, y_train, y_val, "word2vec и tf idf")


word2vec и tf idf - Accuracy: 0.7550
                   precision    recall  f1-score   support

           Бизнес       0.69      0.65      0.67      1125
      Бывший СССР       0.79      0.82      0.80      1520
              Дом       0.86      0.85      0.85      1520
         Из жизни       0.63      0.67      0.65      1520
   Интернет и СМИ       0.74      0.72      0.73      1520
         Культура       0.83      0.85      0.84      1520
              Мир       0.68      0.70      0.69      1520
  Наука и техника       0.77      0.73      0.75      1520
      Путешествия       0.78      0.80      0.79       974
           Россия       0.61      0.56      0.58      1520
Силовые структуры       0.66      0.70      0.68      1520
            Спорт       0.96      0.94      0.95      1520
         Ценности       0.87      0.87      0.87      1181
        Экономика       0.72      0.73      0.73      1520

         accuracy                           0.76     20000
        macro avg

In [238]:
print(f"Улучшение: {acc_w2v_and_tfidf - acc_w2v}")

Улучшение: -0.04490000000000005


Подобный комбинированный метод word2vec + tfidf показал ухудшение качества в сравнении просто с векторизацией с помощью обученной модели word2vec, из-за использования tf idf мы потеряли часть какой-либо информации

# 6. Финально сравним качество всех моделей на тестовой выборке

На обученной word2vec

In [240]:
acc_w2v_test = train_three_models(X_train_w2v, X_test_w2v, y_train, y_test, "train_Word2Vec")

train_Word2Vec - Accuracy: 0.8006
                   precision    recall  f1-score   support

           Бизнес       0.76      0.73      0.74      1125
      Бывший СССР       0.85      0.85      0.85      1520
              Дом       0.89      0.88      0.88      1520
         Из жизни       0.70      0.74      0.72      1520
   Интернет и СМИ       0.77      0.74      0.76      1520
         Культура       0.86      0.87      0.86      1520
              Мир       0.72      0.74      0.73      1520
  Наука и техника       0.82      0.79      0.81      1520
      Путешествия       0.81      0.82      0.82       974
           Россия       0.65      0.63      0.64      1520
Силовые структуры       0.75      0.78      0.77      1520
            Спорт       0.97      0.95      0.96      1521
         Ценности       0.92      0.90      0.91      1180
        Экономика       0.78      0.78      0.78      1520

         accuracy                           0.80     20000
        macro avg   

На rusvec

In [241]:
acc_rusvec = train_three_models(X_train_rusvec, X_test_rusvec, y_train, y_test, "rusvec")

rusvec - Accuracy: 0.0760
                   precision    recall  f1-score   support

           Бизнес       0.00      0.00      0.00      1125
      Бывший СССР       0.00      0.00      0.00      1520
              Дом       0.00      0.00      0.00      1520
         Из жизни       0.00      0.00      0.00      1520
   Интернет и СМИ       0.00      0.00      0.00      1520
         Культура       0.00      0.00      0.00      1520
              Мир       0.08      1.00      0.14      1520
  Наука и техника       0.00      0.00      0.00      1520
      Путешествия       0.00      0.00      0.00       974
           Россия       0.00      0.00      0.00      1520
Силовые структуры       0.00      0.00      0.00      1520
            Спорт       0.00      0.00      0.00      1521
         Ценности       0.00      0.00      0.00      1180
        Экономика       0.00      0.00      0.00      1520

         accuracy                           0.08     20000
        macro avg       0.01

c:\Users\Marsohodik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Marsohodik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Marsohodik\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


На navec

In [242]:
acc_navec_test = train_three_models(X_train_navec, X_test_navec, y_train, y_test, "navec")

navec - Accuracy: 0.7718
                   precision    recall  f1-score   support

           Бизнес       0.66      0.65      0.66      1125
      Бывший СССР       0.83      0.84      0.83      1520
              Дом       0.86      0.87      0.86      1520
         Из жизни       0.66      0.70      0.68      1520
   Интернет и СМИ       0.75      0.73      0.74      1520
         Культура       0.84      0.86      0.85      1520
              Мир       0.71      0.73      0.72      1520
  Наука и техника       0.79      0.76      0.78      1520
      Путешествия       0.78      0.81      0.79       974
           Россия       0.64      0.59      0.62      1520
Силовые структуры       0.68      0.70      0.69      1520
            Спорт       0.96      0.95      0.96      1521
         Ценности       0.91      0.88      0.89      1180
        Экономика       0.75      0.72      0.73      1520

         accuracy                           0.77     20000
        macro avg       0.77 

На обученной word2vec + tfidf

In [245]:
acc_w2v_and_tfidf_test = train_three_models(X_train_w2v_tfidf, X_test_w2v_tfidf, y_train, y_test, "word2vec и tf idf")


word2vec и tf idf - Accuracy: 0.7581
                   precision    recall  f1-score   support

           Бизнес       0.67      0.64      0.66      1125
      Бывший СССР       0.81      0.81      0.81      1520
              Дом       0.86      0.86      0.86      1520
         Из жизни       0.64      0.71      0.68      1520
   Интернет и СМИ       0.74      0.71      0.73      1520
         Культура       0.82      0.85      0.84      1520
              Мир       0.68      0.71      0.70      1520
  Наука и техника       0.78      0.73      0.76      1520
      Путешествия       0.77      0.79      0.78       974
           Россия       0.60      0.56      0.58      1520
Силовые структуры       0.67      0.71      0.69      1520
            Спорт       0.96      0.94      0.95      1521
         Ценности       0.88      0.87      0.88      1180
        Экономика       0.73      0.71      0.72      1520

         accuracy                           0.76     20000
        macro avg

# Подход с обученным word2vec на тестовой выборке также показал самое лучшее качество, чем с взвешиванием tfidf, а также с уже предобученными моделями rusvec и navec. Возможно из-за ввода tfidf мы понизили важность каких-то часто встречающихся, но при этом важных для нашей классификации слов